# Confluent-Kafka

In [2]:
import confluent_kafka
from confluent_kafka import Producer, Consumer, TopicPartition, KafkaError
from confluent_kafka.admin import AdminClient, NewTopic, ConfigResource, NewPartitions
from time import sleep

## AdminClient

In [2]:
# Create an AdminClient
admin_client = AdminClient({
    'bootstrap.servers': 'localhost:9092',  # List your Kafka brokers
    'client.id': 'my_admin_client'
})

### Topics

In [3]:
# Define the topic details
topic_name = "apple"
num_partitions = 1  # Number of partitions for the topic
replication_factor = 1  # Replication factor

# Create the topic object
topic = NewTopic(topic = topic_name, num_partitions = num_partitions, replication_factor = replication_factor)

# Use the admin client to create the topic
# Note: create_topics() returns a dictionary of futures
futures = admin_client.create_topics(new_topics=[topic], validate_only=False)

# Check the result of each topic creation
for topic, future in futures.items():
    try:
        future.result()  # If no exception, the topic was created successfully
        print(f"Topic '{topic}' created successfully")
    except Exception as e:
        print(f"Failed to create topic '{topic}': {e}")

Topic 'apple' created successfully


In [5]:
# get topics
cluster_metadata = admin_client.list_topics()
cluster_metadata.topics

{'__consumer_offsets': TopicMetadata(__consumer_offsets, 50 partitions)}

Some topics management

In [ ]:
admin_client.delete_topics(topics=["apple"])

In [55]:
# check if topics exist (efficiently i.e. not use admin_client.list_topics().topics) and get additional info
topic_collection = confluent_kafka.TopicCollection(["bubu", "quickstart"])
topics_description = admin_client.describe_topics(topics=topic_collection)
for topic, description in topics_description.items():
    if not topics_description[topic].exception():
       topic_partition_info = topics_description[topic].result()
       num_partitions = len(topic_partition_info.partitions)
       print(f"Topic '{topic}' has {num_partitions} partitions")
    else:
        print(f"Failed to describe topic '{topic}': {topics_description[topic].exception()}")

Failed to describe topic 'bubu': KafkaError{code=UNKNOWN_TOPIC_OR_PART,val=3,str="Broker: Unknown topic or partition"}
Topic 'quickstart' has 1 partitions


Configuration management

In [78]:
# ConfigResource represents the Kafka resource (like a topic or broker) whose configuration you want to retrieve or update
config_resource = ConfigResource(ConfigResource.Type.TOPIC, "NordVPN")
# returns a map from each ConfigResource to a collection of configurations.
futures = admin_client.describe_configs([config_resource])

for resource, future in futures.items():
    try:
        config = future.result()  # Get the result for the specific resource
        print(f"Configurations for {resource.name}:")
                
        for config_key, config_value in config.items():
            # Each config_key corresponds to a configuration parameter
            # config_value is a ConfigEntry object that contains value and additional metadata
            print(config_value)

    except Exception as e:
        print(f"Failed to describe config for {resource.name}: {e}")

Configurations for NordVPN:
compression.type="producer"
leader.replication.throttled.replicas=""
message.downconversion.enable="true"
min.insync.replicas="1"
segment.jitter.ms="0"
cleanup.policy="delete"
flush.ms="9223372036854775807"
follower.replication.throttled.replicas=""
segment.bytes="1073741824"
retention.ms="604800000"
flush.messages="9223372036854775807"
message.format.version="2.8-IV1"
max.compaction.lag.ms="9223372036854775807"
file.delete.delay.ms="60000"
max.message.bytes="1048588"
min.compaction.lag.ms="0"
message.timestamp.type="CreateTime"
preallocate="false"
min.cleanable.dirty.ratio="0.5"
index.interval.bytes="4096"
unclean.leader.election.enable="false"
retention.bytes="-1"
delete.retention.ms="86400000"
segment.ms="604800000"
message.timestamp.difference.max.ms="9223372036854775807"
segment.index.bytes="10485760"


In [89]:
# Add partitions to a topic (rarely needed bc each partition has already high throughput)

def add_partitions_to_topic(admin_client, topic_name, new_partition_count):
    # Create NewPartitions object for the topic
    new_partitions = NewPartitions(new_partition_count)
    
    # Send request to add partitions to the topic
    futures = admin_client.create_partitions({topic_name: new_partitions})

    # Process the future result
    for topic, future in futures.items():
        try:
            # Wait for partition creation to finish
            future.result()
            print(f"Partitions added to topic '{topic}' successfully.")
        except Exception as e:
            print(f"Failed to add partitions to topic '{topic}': {e}")


## Producer

In [ ]:
# Create a Kafka producer instance
producer = Producer({
    'bootstrap.servers': 'kafka:9092',
    'api.version.request': True,  # Enable API version request to automatically determine version
})

In [18]:
# Example of sending a message
topic_name = "quickstart"
message_value = "Hello, brother!"

# Optional: Define a callback for delivery confirmation
def delivery_report(err, msg):
    if err is not None:
        print(f"Message delivery failed: {err}")
    else:
        print(f"Message delivered to {msg.topic} [{msg.partition}] at offset {msg.offset}")

# Produce a message, ensure proper serialization before sending
producer.produce(topic_name, value=message_value.encode('utf-8'), callback=delivery_report)

# Wait up to 1 second for events. Call this to ensure delivery reports are received.
producer.flush()

Message delivered to <built-in method topic of cimpl.Message object at 0x7fa7e81ff840> [<built-in method partition of cimpl.Message object at 0x7fa7e81ff840>] at offset <built-in method offset of cimpl.Message object at 0x7fa7e81ff840>


0

In [20]:
# Synchronous send
try:
    # Produce a message
    future = producer.produce('quickstart', value='This is a synchronous message', callback=delivery_report)

    # Wait for the delivery report
    producer.flush(timeout=10)  # This ensures all messages are sent and reports are processed

    # You can check for delivery confirmation by calling the delivery_report callback
    # Note: The delivery report will already have been printed by the callback

except KafkaError as e:
    print(f"Failed to send message: {e}")

Message delivered to <built-in method topic of cimpl.Message object at 0x7fa7e80795c0> [<built-in method partition of cimpl.Message object at 0x7fa7e80795c0>] at offset <built-in method offset of cimpl.Message object at 0x7fa7e80795c0>


In [26]:
# Callback function to handle success
def on_send_success(record_metadata):
    print(f"Message sent successfully to topic: {record_metadata.topic}, "
          f"partition: {record_metadata.partition}, offset: {record_metadata.offset}")

# Callback function to handle errors
def on_send_error(err):
    print(f"Failed to send message: {err}")

# Asynchronous message sending with callback
try:
    # Produce a message
    producer.produce('quickstart', value='This is an asynchronous message', 
                     callback=on_send_success)

    # Flush to ensure all messages are sent and callbacks are processed
    # producer.flush(timeout=10) -> some error

except KafkaError as e:
    print(f"Error while producing message: {e}")

In [ ]:
# Flush the producer to "ensure" all messages are sent and close the producer connection
producer.flush()
producer.close()

## Consumer

In [39]:
# Create a Kafka consumer instance
consumer = Consumer({
    'bootstrap.servers': 'localhost:9092',
    'group.id': 'None',  # Name of the consumer group for dynamic partition assignment
    'auto.offset.reset': 'earliest',  # Start from the earliest message if no valid offset is provided
    'enable.auto.commit': True,  # Enable automatic offset committing
    'auto.commit.interval.ms': 1000,  # Interval for committing offsets
    'partition.assignment.strategy': 'range'
})

# Subscribe to the topic(s)
consumer.subscribe(['cacca'])  # Add as many topics as you want

The first time you call poll() with a
new consumer, it is responsible for finding the GroupCoordinator, joining the consumer group, and receiving a partition assignment. If a rebalance is triggered, it will
be handled inside the poll loop as well, including related callbacks. 

Confluent Kafka defines certain special offset values to represent specific states:
- -1: offset is invalid or not set
- -1000: the consumer's position hasn't been determined yet; possible reasons: 
    - insufficient polling: The consumer hasn't polled enough to join the consumer group and fetch the offsets.
    - incorrect configuration: Misconfiguration in the consumer settings can prevent proper offset retrieval.
    - topic issues: The topic might not have any messages, or there could be issues with the topic's partitions.

In [3]:
output = consumer.poll() # to trigger the automatic partition assigmnet; leave timeout to infinite to make sure assignment, then set it to a small value

topic_partition_cacca = TopicPartition('cacca', 0, 2)  # here the offset is not relevant bc no subsequent poll

# note: the assignment.offset is the offset of the last message that was read and is static. after calling poll it won't change
print(f"assingment info:{consumer.assignment()}") 
print(f"current_offset: {consumer.position([topic_partition_cacca])[0].offset}")

assingment info:[TopicPartition{topic=cacca,partition=0,offset=-1000,leader_epoch=None,error=None}]
current_offset: 3


In [4]:
consumer.seek(topic_partition_cacca) # set the offset to 2 + 1 (next message to consume)
output = consumer.poll(timeout=1.0) # to trigger the offset change
print(f"current_offset: {consumer.position([topic_partition_cacca])[0].offset}")

current_offset: 3


In [20]:
# manual partition assignment -> no automatic rebalancing
# note: it's possible to subscribe in only one way (can't use both subscribe and assign)
partition = 0
offset = 2
topic_quickstart = TopicPartition('quickstart', partition, offset)  
consumer.assign([topic_quickstart]) # Assign any number of partitions to the consumer

In [21]:
output = consumer.poll()
print(consumer.assignment())
print(consumer.position([topic_quickstart])[0].offset)

[TopicPartition{topic=quickstart,partition=0,offset=2,leader_epoch=None,error=None}]
3


In [23]:
consumer.committed([topic_quickstart])

[TopicPartition{topic=quickstart,partition=0,offset=4,leader_epoch=0,error=None}]

In [33]:
# asynchronous commit (default behaviour)
consumer.commit(asynchronous=True)  # set 'enable.auto.commit': False and asynchronous to False to commit synchronously (manual offset commit)

consumers must keep polling Kafka or they will be considered dead and the partitions they are consuming will be handed to another
consumer in the group

contrary to kafka-python, confluent-kafka consumer is not iterable (like Java API), so must use poll()

In [ ]:
# more control over the polling loop
# can be used in a non-blocking manner with a specified timeout

while True:
    msg = consumer.poll(timeout_ms=100)  # Polling with a 100 ms timeout, will block if data is not available in the consumer buffer.
    for tp, messages in msg.items():
        for message in messages:
            print(f'Received message: {message.value}')


In [ ]:
# Create an AdminClient
admin_client = AdminClient({
    'bootstrap.servers': 'localhost:9092',  # List your Kafka brokers
    'client.id': 'my_admin_client'
})